# Week 4 Practice: Optimization Methods

CSCI1409 Deep Learning · based on Lecture 4

## Instructions

- Work in this notebook with Python and NumPy.
- Show your code and its output. Be ready to explain every step.
- Round final answers to 3 decimal places.
- Write short answers to the explanation questions in the **Your answer** cells.

In [1]:
import numpy as np

## Task 1: Gradient descent and the learning rate

We minimize a simple loss with one parameter $w$. Its minimum is at $w = 3$.

$$f(w) = (w - 3)^2, \qquad f'(w) = 2\,(w - 3)$$

Gradient descent repeats this update with learning rate $\eta$:

$$w \leftarrow w - \eta\, f'(w)$$

### 1a) Try different learning rates

1. Start from $w = 0$. Run 10 steps of gradient descent for $\eta = 0.1,\ 0.5,\ 0.9$ and $1.1$. Print $w$ after every step.
2. For each $\eta$, describe what happens: slow, very fast, oscillates, or diverges.

In [5]:
n = [0.1,0.5,0.9,1.1]

for i in range(len(n)):
    w = 0
    for _ in range(10):
        # print(w)
        y = pow(w - 3, 2)
        y_hat = 2 * (w - 3)
        w = w - n[i] * y_hat
        error = y_hat - y
        print(f"error for eta = {n[i]} and w = {w} is {error}")

    


error for eta = 0.1 and w = 0.6000000000000001 is -15
error for eta = 0.1 and w = 1.08 is -10.559999999999999
error for eta = 0.1 and w = 1.464 is -7.5264
error for eta = 0.1 and w = 1.7711999999999999 is -5.431296
error for eta = 0.1 and w = 2.01696 is -3.9675494400000004
error for eta = 0.1 and w = 2.213568 is -2.9324476415999996
error for eta = 0.1 and w = 2.3708544 is -2.191339290624
error for eta = 0.1 and w = 2.49668352 is -1.6541153859993607
error for eta = 0.1 and w = 2.597346816 is -1.2599604390395907
error for eta = 0.1 and w = 2.6778774528 is -0.967435954585338
error for eta = 0.5 and w = 3.0 is -15
error for eta = 0.5 and w = 3.0 is 0.0
error for eta = 0.5 and w = 3.0 is 0.0
error for eta = 0.5 and w = 3.0 is 0.0
error for eta = 0.5 and w = 3.0 is 0.0
error for eta = 0.5 and w = 3.0 is 0.0
error for eta = 0.5 and w = 3.0 is 0.0
error for eta = 0.5 and w = 3.0 is 0.0
error for eta = 0.5 and w = 3.0 is 0.0
error for eta = 0.5 and w = 3.0 is 0.0
error for eta = 0.9 and w = 5.4

**Your answer:**



we can clearly see that learning rate = 0.5 ideally fits with our case because with the second iteration error is 0 and weight doesn't change anymore

for small learning rate (0.1) we see that error decreases but too slow, for learning rate =0.9 error oscillates and for 1.1 error increases with every iteration

### 1b) Explain the results

1. Show that $w_{t+1} - 3 = (1 - 2\eta)\,(w_t - 3)$.
2. Use this formula to explain your results in 1a. For which values of $\eta$ does gradient descent converge?

In [10]:
for i in range(len(n)):
    w = 0
    for _ in range(10):
        # print(w)
        w_old = w
        # y = pow(w - 3, 2)
        y_hat = 2 * (w - 3)
        w = w - n[i] * y_hat
        # error = y_hat - y
        assert abs(w - 3 - (1 - 2*n[i]) * (w_old - 3))< 1e-9
        print("equal")

equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal
equal


**Your answer:**



for small learning rates gradient descent converges to 0.

## Task 2: Mini-batches and momentum

### 2a) Mini-batch gradient

Data: $x = [1, 2, 3, 4]$ and $y = [2, 4, 6, 8]$. Model: $\hat{y} = wx$ with MSE loss. The gradient on a batch of $B$ examples is:

$$\frac{\partial L}{\partial w} = \frac{1}{B} \sum_{i} 2\,(w x_i - y_i)\,x_i$$

1. At $w = 0$, compute the gradient on the full batch (all 4 examples), on batch 1 (first two examples) and on batch 2 (last two examples).
2. How are the three gradients related? Mini-batch gradients are noisy, so why do we use them?

In [14]:
x = np.array([1, 2, 3, 4], dtype=float)
y = np.array([2, 4, 6, 8], dtype=float)

# 1
w = 0

def gradient(w, x, y):
    sum = 0
    B = len(x)
    for i in range(len(x)):
        sum += 2*x[i] * (w*x[i] - y[i])
    grad = sum / B
    return grad
full_grad = gradient(w, x, y)
b_1_gradient = gradient(w, x[:2], y[:2])
b_2_gradient = gradient(w, x[2:], y[2:])
print(full_grad)
print(b_1_gradient)
print(b_2_gradient)


-30.0
-10.0
-50.0


**Your answer:**



full batch is like the mean 
and we see that the smallest gradient is produced by second batch, that means that larger values(x = [3,4] and y = [6,8]) pulls harder 

### 2b) Momentum

Momentum keeps a velocity $v$ (starts at 0) and uses $\beta = 0.9$:

$$v \leftarrow \beta v + g, \qquad w \leftarrow w - \eta v$$

1. The gradient is always $g = 1$. Compute $v$ for 20 steps. What value does $v$ approach?
2. Now the gradient alternates: $+1, -1, +1, -1, \dots$ Compute $v$ for 20 steps. What does momentum do to steps in a steady direction? What does it do to steps that oscillate?

In [2]:
def momentum(gradients)-> list:
    v = 0
    b = 0.9
    history = []
    for g in gradients:
        v = b*v + g
        # w = w - n*v
        history.append(v)
    return history

print(momentum([1]*20))
print(momentum([1 if i%2==0 else -1 for i in range(20)]))


[1.0, 1.9, 2.71, 3.439, 4.0951, 4.68559, 5.217031, 5.6953279000000006, 6.12579511, 6.5132155990000005, 6.861894039100001, 7.175704635190001, 7.458134171671, 7.7123207545039, 7.94108867905351, 8.14697981114816, 8.332281830033345, 8.49905364703001, 8.649148282327008, 8.784233454094307]
[1.0, -0.09999999999999998, 0.91, -0.18099999999999994, 0.8371000000000001, -0.24660999999999988, 0.778051, -0.2997540999999999, 0.7302213100000001, -0.34280082099999987, 0.6914792611000001, -0.37766866500999985, 0.6600982014910002, -0.40591161865809977, 0.6346795432077101, -0.4287884111130609, 0.6140904299982453, -0.44731861300157927, 0.5974132482985787, -0.46232807653127916]


**Your answer:**



momentum helps to deal with the oscillation problem, because momentum works like memory, it stores the history of past gradients. 

when signal has the same direction(sign), memory amplifies the effect and when sign is different it negates the effect